# Chapter 09 — SQL Fundamentals and MySQL Connectivity

SQL is the language for relational data. Connect relational design and query logic to Python database drivers, transactions, parameterized SQL, and safe resource handling.

## Prerequisites
Basic tables and Python functions. MySQL server only needed for live-connection exercises; SQLite is used for local SQL drills.

## Learning goals
- Explain tables, rows, primary keys, and foreign keys.
- Write SELECT, WHERE, GROUP BY, HAVING, JOIN, and subqueries.
- Connect with mysql-connector-python or PyMySQL.
- Use parameterized statements, transactions, cursors, commits, and rollbacks.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Practical scenario

Practice SQL locally, then adapt the same query logic to a MySQL database using a real driver.

## Tasks to complete

- Create customer and order tables with primary and foreign keys.
- Insert rows using parameterized statements.
- Write a JOIN grouped by city.
- Show how to rollback an update.
- Explain SQLite ? placeholders versus MySQL driver %s placeholders.

**Working rule:** write your own code in the next cell before opening the reference solution. Explain each assumption and validate the result.

In [2]:
# TODO: complete the practical lab here.
# Start by loading the relevant dataset or defining the required input.
# Add validation checks before calculating the requested outputs.

## Worked reference solution

The following is one valid approach. It is not the only possible solution. Compare your work with it, and explain any differences in assumptions, handling, or output.

In [3]:
import sqlite3
con=sqlite3.connect(":memory:"); con.execute("PRAGMA foreign_keys=ON")
con.executescript("CREATE TABLE customers(customer_id TEXT PRIMARY KEY,city TEXT NOT NULL); CREATE TABLE orders(order_id TEXT PRIMARY KEY,customer_id TEXT NOT NULL REFERENCES customers(customer_id),revenue REAL NOT NULL CHECK(revenue>=0));")
con.executemany("INSERT INTO customers VALUES (?,?)",[("C1","Kolkata"),("C2","Delhi")]); con.executemany("INSERT INTO orders VALUES (?,?,?)",[("O1","C1",1200),("O2","C1",800),("O3","C2",1500)])
con.commit()
report=con.execute("SELECT c.city,COUNT(*) AS orders,SUM(o.revenue) AS revenue FROM customers c JOIN orders o USING(customer_id) GROUP BY c.city ORDER BY revenue DESC").fetchall()
print(report); con.execute("UPDATE orders SET revenue=? WHERE order_id=?",(9999,"O1")); con.rollback(); print("O1 after rollback:",con.execute("SELECT revenue FROM orders WHERE order_id=?",("O1",)).fetchone()[0]); con.close()

[('Kolkata', 2, 2000.0), ('Delhi', 1, 1500.0)]
O1 after rollback: 1200.0


## Extension challenge

Implement an idempotent customer import using a unique business key. Compare MySQL `INSERT`, `INSERT IGNORE`, and `INSERT ... ON DUPLICATE KEY UPDATE` and test duplicate handling.

## Further reading

- MySQL Reference: https://dev.mysql.com/doc/refman/8.4/en/
- Connector/Python: https://dev.mysql.com/doc/connector-python/en/
- PyMySQL: https://pymysql.readthedocs.io/

## Completion checklist

- [ ] I can explain the topic without reading the cells.
- [ ] I can reproduce the examples from a clean kernel.
- [ ] I completed each task and checked the reference solution.
- [ ] I tested an edge case.
- [ ] I can describe at least one limitation or risk.